In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import shutil
import random
import numpy as np
from pathlib import Path

from torch_pointcloud.utils.io import load_off, save_off

## ModelNet

### ModelNet10

In [11]:
def create_mock_dataset(data_dir: str, mock_dir: str, num_samples_per_class: int = 2, num_points: int = 100):
    data_dirpath = Path(data_dir)
    mock_dirpath = Path(mock_dir)
    
    classes = [path.name for path in data_dirpath.iterdir() if path.is_dir()]

    mock_dirpath.mkdir(parents=True, exist_ok=True)

    for class_name in classes:
        off_files = list(Path(data_dirpath, class_name, "test").rglob("*.off"))

        num_samples = min(num_samples_per_class, len(off_files))
        selected_files = random.sample(off_files, num_samples)

        for off_file in selected_files:
            # Load the original .off file
            vertices, faces = load_off(off_file)

            # Randomly subsample points (vertices)
            if vertices.shape[0] > num_points:
                subsample_indices = np.random.choice(vertices.shape[0], num_points, replace=False)
                vertices = vertices[subsample_indices]

            out_path = mock_dirpath / Path(off_file).relative_to(data_dirpath)
            out_path.parent.mkdir(parents=True, exist_ok=True)
            save_off(out_path, vertices, faces)

In [12]:
create_mock_dataset(
    "../data/ModelNet10/raw",
    "data/ModelNet10/raw",
    num_samples_per_class=1,
    num_points=100
)

Mock dataset created at data/ModelNet10/raw


In [4]:
from torch_pointcloud.datasets.modelnet import ModelNet10

In [7]:
dataset = ModelNet10(
    root="mocks",
    train=True,
)

Processing: 100%|██████████| 10/10 [00:00<00:00, 33.48it/s]


In [8]:
dataset[0]

{'xyz': tensor([[ -1.9456,   0.1524, -10.9477],
         [ -4.9104,  28.8727, -12.8899],
         [  7.3225,   0.6101,  -8.7475],
         ...,
         [ 13.7046,  30.1631, -13.8314],
         [ -4.2977,   0.6115, -12.1432],
         [ 14.7524, -28.9370,   3.7513]]),
 'face': tensor([[    0,     1,     2],
         [    1,     0,     3],
         [    8,     3,     0],
         ...,
         [10171, 10173, 10179],
         [10173, 10175, 10179],
         [10177, 10179, 10175]]),
 'target': tensor([7])}

In [14]:
import torch

import torch_pointcloud.transforms.functional as F

In [15]:
data = {"xyz": torch.rand(100, 3), "feature": torch.rand(100, 6)}
num_samples = 10

sampled_data, indices = F.random_sample_data(data, num_samples)
# Check that the shapes are correct
assert sampled_data["xyz"].shape[0] == num_samples, "'xyz' should have num_samples rows"
assert sampled_data["feature"].shape[0] == num_samples, "'feature' should have num_samples rows"
assert indices.shape[0] == num_samples, "Indices should have num_samples values"

# Check that the indices are within valid range
assert torch.all(indices >= 0) and torch.all(indices < data["xyz"].size(0))

AssertionError: 

In [16]:
indices

tensor([78, 35, 17, 12, 52, 98, 15, 91, 86, 31])

In [17]:
data["xyz"].size(0)

10

In [19]:
data = {"xyz": torch.rand(100, 3), "feature": torch.rand(100, 6)}
data["xyz"]

tensor([[8.7265e-01, 3.9589e-01, 5.5618e-01],
        [5.4136e-01, 5.7564e-01, 1.5211e-01],
        [9.3114e-01, 9.5996e-02, 7.6875e-01],
        [4.5294e-01, 7.0735e-01, 7.1790e-01],
        [3.3067e-01, 9.6745e-01, 6.7555e-01],
        [5.0855e-01, 6.6909e-01, 8.6137e-01],
        [7.2417e-01, 7.2132e-01, 4.8635e-02],
        [5.9251e-01, 1.5263e-01, 1.0571e-01],
        [5.9794e-01, 3.7285e-01, 9.0699e-02],
        [4.3202e-01, 9.0430e-01, 9.5459e-01],
        [3.5768e-01, 3.8248e-01, 1.6108e-01],
        [3.6594e-01, 4.3560e-01, 6.6719e-02],
        [1.8409e-01, 6.1235e-01, 1.9115e-02],
        [1.5596e-01, 4.8030e-01, 4.5877e-01],
        [4.0480e-01, 4.1708e-01, 5.9629e-01],
        [9.4216e-01, 7.8622e-01, 2.4365e-01],
        [8.2294e-01, 9.5357e-01, 9.5574e-01],
        [5.6010e-01, 2.7585e-01, 9.5307e-01],
        [6.6048e-01, 5.9004e-02, 3.3586e-01],
        [4.2765e-01, 6.3252e-01, 8.6520e-01],
        [1.2347e-01, 6.1439e-01, 2.9370e-01],
        [4.6033e-01, 4.9275e-01, 4